# Import packages

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import PIL 

# Earthworm area estimator code

In [ ]:
# A4 paper dimensions in millimeters
A4_W_MM = 210.0
A4_H_MM = 297.0

def order_points(pts):
    """
    Orders 4 points in the order: top-left, top-right, bottom-right, bottom-left.
    """
    pts = np.asarray(pts, dtype=np.float32)
    s = pts.sum(axis=1)
    d = np.diff(pts, axis=1).ravel()

    top_left = pts[np.argmin(s)]
    bottom_right = pts[np.argmax(s)]
    top_right = pts[np.argmin(d)]
    bottom_left = pts[np.argmax(d)]

    return np.array([top_left, top_right, bottom_right, bottom_left], dtype=np.float32)


def four_point_transform(image, pts, output_width=2480, output_height=3508):
    """
    Applies a perspective transform to obtain a top-down view of the A4 paper.
    """
    rect = order_points(pts)
    dst = np.array(
        [
            [0, 0],
            [output_width - 1, 0],
            [output_width - 1, output_height - 1],
            [0, output_height - 1],
        ],
        dtype=np.float32,
    )
    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (output_width, output_height))
    return warped, rect, M


def detect_a4_contour_robust(image):
    """
    Detects the largest quadrilateral in the image that matches the A4 aspect ratio.
    Enhanced with multi-channel edge detection for robustness to different backgrounds.
    """
    # Multi-strategy edge detection for different backgrounds
    
    # Strategy 1: Standard grayscale approach
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur_gray = cv2.GaussianBlur(gray, (5, 5), 0)
    edges_gray = cv2.Canny(blur_gray, 50, 150)
    
    # Strategy 2: HSV saturation channel - works well for white paper on colored backgrounds
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    s_channel = hsv[:, :, 1]
    blur_s = cv2.GaussianBlur(s_channel, (5, 5), 0)
    # Invert for white paper (low saturation)
    s_inv = 255 - s_channel
    edges_sat = cv2.Canny(s_inv, 30, 100)
    
    # Strategy 3: LAB L-channel for better contrast on varied backgrounds
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l_channel = lab[:, :, 0]
    blur_l = cv2.GaussianBlur(l_channel, (5, 5), 0)
    edges_lab = cv2.Canny(blur_l, 50, 150)
    
    # Combine all edge detection strategies
    edges_combined = cv2.bitwise_or(edges_gray, edges_sat)
    edges_combined = cv2.bitwise_or(edges_combined, edges_lab)
    
    # Morphological operations to clean up edges
    edges_combined = cv2.dilate(edges_combined, np.ones((3, 3), np.uint8), iterations=1)
    edges_combined = cv2.morphologyEx(edges_combined, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))

    contours, _ = cv2.findContours(edges_combined, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)

    img_area = image.shape[0] * image.shape[1]
    best_quad = None
    best_area = 0

    for c in contours[:30]:
        area = cv2.contourArea(c)
        if area < 0.1 * img_area:
            continue

        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)

        if len(approx) == 4 and cv2.isContourConvex(approx):
            pts = approx.reshape(4, 2).astype(np.float32)
            rect = order_points(pts)

            w_top = np.linalg.norm(rect[1] - rect[0])
            w_bottom = np.linalg.norm(rect[2] - rect[3])
            h_left = np.linalg.norm(rect[3] - rect[0])
            h_right = np.linalg.norm(rect[2] - rect[1])

            w = 0.5 * (w_top + w_bottom)
            h = 0.5 * (h_left + h_right)
            if min(w, h) <= 0:
                continue

            ratio = max(w, h) / max(1e-6, min(w, h))
            a4_ratio = A4_H_MM / A4_W_MM

            if abs(ratio - a4_ratio) < 0.35 and area > best_area:
                best_quad = pts
                best_area = area

    if best_quad is None:
        raise RuntimeError("Could not detect an A4-like quadrilateral.")

    return order_points(best_quad), edges_combined


def contour_to_box_points(c):
    rect = cv2.minAreaRect(c)
    box = cv2.boxPoints(rect)
    return order_points(box.astype(np.float32))


def score_a4_candidate(pts, area, img_area):
    rect = order_points(pts)

    w_top = np.linalg.norm(rect[1] - rect[0])
    w_bottom = np.linalg.norm(rect[2] - rect[3])
    h_left = np.linalg.norm(rect[3] - rect[0])
    h_right = np.linalg.norm(rect[2] - rect[1])

    w = 0.5 * (w_top + w_bottom)
    h = 0.5 * (h_left + h_right)

    if min(w, h) <= 1:
        return None

    ratio = max(w, h) / min(w, h)
    a4_ratio = A4_H_MM / A4_W_MM  # ~1.414
    ratio_err = abs(ratio - a4_ratio)

    # prefer big contours with near-A4 ratio
    area_frac = area / img_area
    score = area_frac - 0.8 * ratio_err

    return {
        "rect": rect,
        "score": score,
        "ratio": ratio,
        "ratio_err": ratio_err,
        "area": area,
        "area_frac": area_frac,
    }


def detect_a4_contour(image, debug=False):
    """
    Robust A4 detection on light/dark/colored desks.
    Strategy:
      1. Build several candidate masks / edge maps.
      2. Find large contours.
      3. First try approxPolyDP 4-point contour.
      4. Fallback to minAreaRect box if approx is not clean.
      5. Keep best A4-like candidate by score.
    """
    img_area = image.shape[0] * image.shape[1]

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)

    s = hsv[:, :, 1]
    l = lab[:, :, 0]

    gray_blur = cv2.GaussianBlur(gray, (5, 5), 0)
    l_blur = cv2.GaussianBlur(l, (5, 5), 0)

    # 1) Edge-based masks
    edges1 = cv2.Canny(gray_blur, 50, 150)
    edges2 = cv2.Canny(l_blur, 40, 120)

    # 2) Bright-paper masks
    # Otsu on LAB lightness often works better than raw grayscale for paper/background separation
    _, th_l_otsu = cv2.threshold(l_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # White paper often has low saturation; use inverse saturation as another cue
    s_inv = 255 - s
    _, th_s = cv2.threshold(s_inv, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    candidate_maps = [
        edges1,
        edges2,
        th_l_otsu,
        th_s,
        cv2.bitwise_or(edges1, edges2),
        cv2.bitwise_or(th_l_otsu, th_s),
    ]

    best = None
    debug_map = None

    for candidate in candidate_maps:
        work = candidate.copy()

        # unify broken page boundaries / suppress holes
        work = cv2.morphologyEx(work, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8), iterations=2)
        work = cv2.dilate(work, np.ones((3, 3), np.uint8), iterations=1)

        contours, _ = cv2.findContours(work, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = sorted(contours, key=cv2.contourArea, reverse=True)[:50]

        for c in contours:
            area = cv2.contourArea(c)
            if area < 0.08 * img_area:
                continue

            peri = cv2.arcLength(c, True)
            approx = cv2.approxPolyDP(c, 0.02 * peri, True)

            candidates = []

            # Route A: exact-ish quadrilateral
            if len(approx) == 4 and cv2.isContourConvex(approx):
                pts = approx.reshape(4, 2).astype(np.float32)
                candidates.append(pts)

            # Route B: fallback rotated rectangle from contour
            box_pts = contour_to_box_points(c)
            candidates.append(box_pts)

            for pts in candidates:
                scored = score_a4_candidate(pts, area, img_area)
                if scored is None:
                    continue

                # fairly tolerant because perspective + imperfect contours can distort ratio
                if scored["ratio_err"] > 0.55:
                    continue

                # avoid boxes that almost cover the whole image unless strongly plausible
                if scored["area_frac"] > 0.98:
                    continue

                if best is None or scored["score"] > best["score"]:
                    best = scored
                    debug_map = work.copy()

    if best is None:
        raise RuntimeError(
            "Could not detect an A4-like region. Try tighter crop, better lighting, or looser thresholds."
        )

    return best["rect"], debug_map

def detect_objects_on_rectified_paper(
    rectified_img,
    dark_threshold=200,
    min_object_pixels=100,
    method="adaptive",
):
    """
    Detects dark objects on the rectified A4 paper and measures their area,
    with robustness to shadows via illumination normalization + adaptive/otsu/global thresholding.
    """
    gray = cv2.cvtColor(rectified_img, cv2.COLOR_BGR2GRAY)

    # Illumination normalization with larger kernel for better shadow handling
    blur_size = 201
    blur = cv2.GaussianBlur(gray, (blur_size, blur_size), 0)
    blur = np.maximum(blur, 1)

    norm_float = (gray.astype(np.float32) / blur.astype(np.float32)) * 128.0
    norm = np.clip(norm_float, 0, 255).astype(np.uint8)

    non_white_pixels_gray = int(np.sum(norm < dark_threshold))

    if method == "global":
        _, binary_mask = cv2.threshold(norm, dark_threshold, 255, cv2.THRESH_BINARY_INV)
    elif method == "otsu":
        _, binary_mask = cv2.threshold(norm, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    else:
        binary_mask = cv2.adaptiveThreshold(
            norm,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY_INV,
            blockSize=51,
            C=5,
        )

    kernel = np.ones((3, 3), np.uint8)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel, iterations=1)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    clean_mask = np.zeros_like(binary_mask)
    object_count = 0
    kept_pixel_area = 0

    h_img, w_img = rectified_img.shape[:2]

    for c in contours:
        area = cv2.contourArea(c)
        if area < min_object_pixels:
            continue

        x, y, w, h = cv2.boundingRect(c)
        aspect_ratio = max(w, h) / max(1, min(w, h))

        if aspect_ratio > 10:
            continue

        if area > 0.8 * (h_img * w_img):
            continue

        cv2.drawContours(clean_mask, [c], -1, 255, thickness=-1)
        object_count += 1
        kept_pixel_area += area

    covered_pixels = int(np.sum(clean_mask > 0))

    h, w = rectified_img.shape[:2]
    mm_per_pixel_x = A4_W_MM / w
    mm_per_pixel_y = A4_H_MM / h
    mm2_per_pixel = mm_per_pixel_x * mm_per_pixel_y

    covered_area_mm2 = covered_pixels * mm2_per_pixel
    covered_area_cm2 = covered_area_mm2 / 100.0

    overlay = rectified_img.copy()
    overlay[clean_mask > 0] = (0, 0, 255)
    vis = cv2.addWeighted(rectified_img, 0.7, overlay, 0.3, 0)

    return {
        "gray_image": gray,
        "normalized_gray": norm,
        "raw_non_white_pixels_gray": non_white_pixels_gray,
        "binary_mask": binary_mask,
        "clean_mask": clean_mask,
        "overlay": vis,
        "object_count": object_count,
        "covered_pixels": covered_pixels,
        "covered_area_mm2": covered_area_mm2,
        "covered_area_cm2": covered_area_cm2,
        "mm_per_pixel_x": mm_per_pixel_x,
        "mm_per_pixel_y": mm_per_pixel_y,
        "mm2_per_pixel": mm2_per_pixel,
    }


def rectify_and_measure(
    input_path,
    rectified_path="rectified_a4.jpg",
    debug_path="debug_detected.jpg",
    mask_path="objects_mask.png",
    overlay_path="objects_overlay.jpg",
    dpi=300,
    dark_threshold=200,
    min_object_pixels=100,
    method="adaptive",
    return_plot=False,
):
    """
    Full pipeline: detects A4, rectifies, detects objects, measures area, and saves results.
    """
    image = cv2.imread(str(input_path))
    if image is None:
        raise FileNotFoundError(f"Could not read image: {input_path}")

    corners, _ = detect_a4_contour(image)

    output_width = int(round((A4_W_MM / 25.4) * dpi))
    output_height = int(round((A4_H_MM / 25.4) * dpi))

    rectified, rect, M = four_point_transform(
        image,
        corners,
        output_width=output_width,
        output_height=output_height,
    )

    debug = image.copy()
    cv2.polylines(debug, [rect.astype(int)], isClosed=True, color=(0, 255, 0), thickness=4)
    for i, p in enumerate(rect.astype(int)):
        cv2.circle(debug, tuple(p), 10, (0, 0, 255), -1)
        cv2.putText(
            debug,
            str(i),
            tuple(p + 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 0, 0),
            2,
        )

    objects = detect_objects_on_rectified_paper(
        rectified,
        dark_threshold=dark_threshold,
        min_object_pixels=min_object_pixels,
        method=method,
    )

    cv2.imwrite(str(rectified_path), rectified)
    cv2.imwrite(str(debug_path), debug)
    cv2.imwrite(str(mask_path), objects["clean_mask"])
    cv2.imwrite(str(overlay_path), objects["overlay"])

    result = {
        "corners": rect,
        "transform_matrix": M,
        "output_size_px": (output_width, output_height),
        "raw_non_white_pixels_gray": objects["raw_non_white_pixels_gray"],
        "object_count": objects["object_count"],
        "covered_pixels": objects["covered_pixels"],
        "covered_area_mm2": objects["covered_area_mm2"],
        "covered_area_cm2": objects["covered_area_cm2"],
        "mm_per_pixel_x": objects["mm_per_pixel_x"],
        "mm_per_pixel_y": objects["mm_per_pixel_y"],
        "mm2_per_pixel": objects["mm2_per_pixel"],
        "rectified_path": str(rectified_path),
        "debug_path": str(debug_path),
        "mask_path": str(mask_path),
        "overlay_path": str(overlay_path),
    }

    if return_plot:
        fig, axs = plt.subplots(1, 3, figsize=(15, 5))

        axs[0].imshow(cv2.cvtColor(objects["overlay"], cv2.COLOR_BGR2RGB))
        axs[0].set_title("Detected Objects Overlay")
        axs[0].axis("off")

        axs[1].imshow(objects["gray_image"], cmap="gray")
        axs[1].set_title("Original grayscale")
        axs[1].axis("off")

        axs[2].imshow(objects["normalized_gray"], cmap="gray")
        axs[2].set_title("Normalized grayscale (shadow reduced)")
        axs[2].axis("off")

        plt.tight_layout()
        result["plot"] = fig

    return result

# Example 

In [ ]:
file_path = "/path/to/your/image.jpg"

img_pil = PIL.Image.open(file_path)
info = img_pil.info

dpi = info.get('dpi', None)
print("DPI from metadata:", dpi)

result = rectify_and_measure(
    input_path=file_path,
    rectified_path="rectified_a4.jpg",
    debug_path="debug_detected.jpg",
    mask_path="objects_mask.png",
    overlay_path="objects_overlay.jpg",
    dpi=dpi[0] if dpi is not None else 300,  # use metadata DPI if available, else default to 300
    dark_threshold=400,
    min_object_pixels=1000/0.5,  # ignore anything < 50 mm^2 in pixels (assuming 0.5 mm^2 per pixel)
    # method="adaptive",    
    return_plot=True,
)

print("Detected objects:", result["object_count"])
print("Covered pixels:", result["covered_pixels"])
print("Covered area (mm^2):", result["covered_area_mm2"])
print("Covered area (cm^2):", result["covered_area_cm2"])
print("Mask saved to:", result["mask_path"])
print("Overlay saved to:", result["overlay_path"])